# Notebook 2: Renaming the image and annotation files

This Jupyter notebook renames the selected image tiles and their corresponding annotation files to sequential numbering. Files are matched across folders by stem name.

The script was written for a project commissioned by the Municipality of Wageningen in relation to the course Remote Sensing and GIS Integration (GRS60312).

The authors of the script are Polly Cheung, Pascal Dubbelman, Anthony Jansen, Iris Lagemaat, and Susanna van de Wetering.

Last edited: 17/06/2026

Before starting the script it is important that the image files (.tif and .tfw) are stored in a folder on Google Drive called "images", and the corresponding annotation files (.xml) are stored in a folder called "labels".

In [1]:
# Connect with Google Drive
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────

# Folder containing your .tif and .tfw files
IMAGE_FOLDER = r"/content/drive/My Drive/images"

# Folder containing your .xml annotation files
XML_FOLDER = r"/content/drive/My Drive/labels"

# Output folders (can be same as input for in-place rename, or different to copy)
OUTPUT_IMAGE_FOLDER = r"/content/drive/My Drive/TestImages"
OUTPUT_XML_FOLDER   = r"/content/drive/My Drive/TestLabels"

# Starting number (e.g. 1 → files become 1.tif, 2.tif, ...)
START_NUMBER = 1

# Sort order before numbering
# 'name'  → alphabetical by original filename (most predictable)
# 'mtime' → by file modification time, oldest first
SORT_BY = "name"

# True = preview only, no files changed
# False = actually rename/copy files
DRY_RUN = False

In [ ]:
# Install packages
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

IMAGE_EXTS = {".tif", ".tfw"}
XML_EXT    = ".xml"

image_folder        = Path(IMAGE_FOLDER)
xml_folder          = Path(XML_FOLDER)
output_image_folder = Path(OUTPUT_IMAGE_FOLDER)
output_xml_folder   = Path(OUTPUT_XML_FOLDER)

for folder in [image_folder, xml_folder]:
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

if not DRY_RUN:
    output_image_folder.mkdir(parents=True, exist_ok=True)
    output_xml_folder.mkdir(parents=True, exist_ok=True)

# ── collect stems from image folder ──────────────────────────────────────────
image_stems = defaultdict(dict)  # stem → {ext: path}
for f in image_folder.iterdir():
    if f.suffix.lower() in IMAGE_EXTS:
        image_stems[f.stem][f.suffix.lower()] = f

# ── collect stems from xml folder ────────────────────────────────────────────
xml_stems = {}  # stem → path
for f in xml_folder.iterdir():
    if f.suffix.lower() == XML_EXT:
        xml_stems[f.stem] = f

# ── find matching stems ───────────────────────────────────────────────────────
all_stems   = sorted(set(image_stems.keys()) | set(xml_stems.keys()))
matched     = [s for s in all_stems if s in image_stems and s in xml_stems]
img_only    = [s for s in all_stems if s in image_stems and s not in xml_stems]
xml_only    = [s for s in all_stems if s in xml_stems and s not in image_stems]

if SORT_BY == "mtime":
    def get_mtime(stem):
        files = image_stems.get(stem, {})
        f = files.get(".tif") or (list(files.values())[0] if files else xml_stems.get(stem))
        return f.stat().st_mtime
    matched  = sorted(matched,  key=get_mtime)
    img_only = sorted(img_only, key=get_mtime)
    xml_only = sorted(xml_only, key=get_mtime)

print(f"Image folder : {image_folder}")
print(f"XML folder   : {xml_folder}")
print(f"Matched stems (tif+tfw+xml): {len(matched)}")
print(f"Image only (no xml)        : {len(img_only)}")
print(f"XML only (no tif/tfw)      : {len(xml_only)}")
print(f"Dry run: {DRY_RUN}")

if img_only:
    print(f"\n⚠️  Stems with image but no XML:")
    for s in img_only[:10]:
        print(f"   {s}")
    if len(img_only) > 10:
        print(f"   ... and {len(img_only)-10} more")

if xml_only:
    print(f"\n⚠️  Stems with XML but no image:")
    for s in xml_only[:10]:
        print(f"   {s}")
    if len(xml_only) > 10:
        print(f"   ... and {len(xml_only)-10} more")

In [ ]:
# ── BUILD RENAME PLAN ─────────────────────────────────────────────────────────
# Only matched stems are renumbered. img_only and xml_only are reported but skipped.

plan = []  # list of dicts with old/new paths for each file in the group

for i, stem in enumerate(matched):
    new_stem = str(START_NUMBER + i)
    group    = []

    # image files (.tif, .tfw)
    for ext, old_path in sorted(image_stems[stem].items()):
        new_path = output_image_folder / f"{new_stem}{ext}"
        group.append({"old": old_path, "new": new_path, "type": "image"})

    # xml file
    old_xml  = xml_stems[stem]
    new_xml  = output_xml_folder / f"{new_stem}.xml"
    new_tif  = f"{new_stem}.tif"
    group.append({"old": old_xml, "new": new_xml, "type": "xml", "new_tif": new_tif})

    plan.append(group)

# ── PREVIEW ───────────────────────────────────────────────────────────────────
print(f"{'OLD':<50} → NEW")
print("-" * 75)

preview_groups = plan[:7] + ([None] if len(plan) > 10 else []) + (plan[-3:] if len(plan) > 10 else [])
for group in preview_groups:
    if group is None:
        print(f"  ... ({len(plan) - 10} more groups) ...")
        continue
    for item in group:
        print(f"{item['old'].name:<50} → {item['new'].name}")
    print()

# conflict check
all_new = [str(item["new"]) for group in plan for item in group]
if len(all_new) != len(set(all_new)):
    print("⚠️  WARNING: duplicate output names detected!")
else:
    print(f"✓  No naming conflicts. {len(plan)} groups ({len(all_new)} files) ready to rename.")

In [ ]:
# ── EXECUTE ───────────────────────────────────────────────────────────────────
# Set DRY_RUN = False in Settings and re-run all cells to apply.

if DRY_RUN:
    print("DRY RUN — no files were changed.")
    print("Set DRY_RUN = False in the Settings cell and re-run to apply.")
else:
    renamed = 0
    errors  = []
    in_place_image = image_folder.resolve() == output_image_folder.resolve()
    in_place_xml   = xml_folder.resolve()   == output_xml_folder.resolve()

    for group in plan:
        for item in group:
            try:
                old_path = item["old"]
                new_path = item["new"]

                if item["type"] == "xml":
                    # parse, update <filename> and <path>, write
                    tree = ET.parse(old_path)
                    root = tree.getroot()

                    fn = root.find("filename")
                    if fn is not None:
                        fn.text = item["new_tif"]

                    pth = root.find("path")
                    if pth is not None:
                        pth.text = str(output_image_folder / item["new_tif"])

                    tree.write(new_path, encoding="utf-8", xml_declaration=True)
                    if in_place_xml and old_path.resolve() != new_path.resolve():
                        old_path.unlink()

                else:
                    if in_place_image:
                        old_path.rename(new_path)
                    else:
                        shutil.copy2(old_path, new_path)

                renamed += 1

            except Exception as e:
                errors.append((item["old"], str(e)))

    print(f"✓  Renamed {renamed} files across {len(plan)} groups.")
    if errors:
        print(f"\n⚠️  {len(errors)} errors:")
        for path, err in errors:
            print(f"   {path.name}: {err}")

In [ ]:
# ── VERIFY ────────────────────────────────────────────────────────────────────
# Run after execution to confirm output is complete and consistent.

if not DRY_RUN:
    img_out_stems = defaultdict(set)
    for f in output_image_folder.iterdir():
        if f.suffix.lower() in IMAGE_EXTS:
            img_out_stems[f.stem].add(f.suffix.lower())

    xml_out_stems = {f.stem for f in output_xml_folder.iterdir() if f.suffix.lower() == XML_EXT}

    issues = []
    for stem in sorted(set(img_out_stems) | xml_out_stems):
        exts   = img_out_stems.get(stem, set())
        has_tif = ".tif" in exts
        has_tfw = ".tfw" in exts
        has_xml = stem in xml_out_stems
        if not (has_tif and has_tfw and has_xml):
            missing = [e for e, v in [(".tif", has_tif), (".tfw", has_tfw), (".xml", has_xml)] if not v]
            issues.append(f"{stem}: missing {', '.join(missing)}")

    total = len(set(img_out_stems) | xml_out_stems)
    if not issues:
        print(f"✓  All {total} stems have complete .tif / .tfw / .xml triplets.")
    else:
        print(f"⚠️  {len(issues)} incomplete triplets:")
        for issue in issues:
            print(f"   {issue}")